In [47]:
# ==========================================================
# UNIVERSAL QA CODE (AUTO COLUMN DETECT + ROUGE SCORE)
# Works on most Kaggle QA CSV datasets
# ==========================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline
from sklearn.model_selection import train_test_split

# ==========================================================
# LOAD DATA
# ==========================================================

train_path = "/kaggle/input/datasets/rashiagarwal12/q-and-a-dataset/qa_train.csv"
test_path  = "/kaggle/input/datasets/rashiagarwal12/q-and-a-dataset/qa_test_with_labels.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train Columns:", train_df.columns.tolist())
print("Test Columns :", test_df.columns.tolist())

# ==========================================================
# AUTO COLUMN DETECTION
# ==========================================================

def detect_column(df, keywords):
    for key in keywords:
        for col in df.columns:
            if key in col.lower():
                return col
    return None

id_col = detect_column(train_df, ["id"])

question_col = detect_column(train_df, [
    "question","query","prompt","instruction","input"
])

context_col = detect_column(train_df, [
    "context","paragraph","passage","article","text","story","document"
])

target_col = detect_column(train_df, [
    "answer","response","output","label","target"
])

# fallback
if question_col is None:
    question_col = train_df.columns[1]

if context_col is None:
    context_col = question_col

if target_col is None:
    target_col = train_df.columns[-1]

if id_col is None:
    id_col = train_df.columns[0]

print("\nDetected Columns:")
print("ID       :", id_col)
print("Question :", question_col)
print("Context  :", context_col)
print("Target   :", target_col)

# ==========================================================
# VALIDATION SPLIT
# ==========================================================

train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

# ==========================================================
# LOAD QA MODEL (GOOD SCORE)
# ==========================================================

qa = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2"
)

# ==========================================================
# SIMPLE ROUGE-L FUNCTION
# ==========================================================

def lcs(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l(pred, ref):
    pred_words = str(pred).split()
    ref_words  = str(ref).split()

    if len(pred_words)==0 or len(ref_words)==0:
        return 0

    common = lcs(pred_words, ref_words)

    prec = common / len(pred_words)
    rec  = common / len(ref_words)

    if prec + rec == 0:
        return 0

    return (2*prec*rec)/(prec+rec)

# ==========================================================
# VALIDATION SCORE
# ==========================================================

scores = []

print("\nRunning Validation...")

for _, row in val_part.head(100).iterrows():   # fast scoring

    q = str(row[question_col])
    c = str(row[context_col])
    true_ans = str(row[target_col])

    try:
        pred = qa(question=q, context=c)["answer"]
    except:
        pred = ""

    score = rouge_l(pred, true_ans)
    scores.append(score)

print("\nValidation ROUGE-L :", np.mean(scores))

# ==========================================================
# TEST PREDICTIONS
# ==========================================================

outputs = []

print("\nGenerating Test Predictions...")

for _, row in test_df.iterrows():

    q = str(row[question_col]) if question_col in test_df.columns else str(row.iloc[1])

    if context_col in test_df.columns:
        c = str(row[context_col])
    else:
        c = q

    try:
        pred = qa(question=q, context=c)["answer"]
    except:
        pred = ""

    outputs.append(pred)

# ==========================================================
# SUBMISSION
# ==========================================================

submission = pd.DataFrame()

if id_col in test_df.columns:
    submission[id_col] = test_df[id_col]
else:
    submission["id"] = range(len(test_df))

submission["output"] = outputs

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv created successfully!")

Train Columns: ['id', 'input', 'response']
Test Columns : ['id', 'input', 'response']

Detected Columns:
ID       : id
Question : input
Context  : input
Target   : response


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Validation...

Validation ROUGE-L : 0.20303030303030303

Generating Test Predictions...

submission.csv created successfully!
